# Reproducing Choi et al. (2025) from dandiset 002014

Paper: Choi, Amjad, Murray, ... and Schwerdt (2025), "Aseptic, semi-sealed cranial chamber implants
for chronic multi-channel neurochemical and electrophysiological neural recording in nonhuman
primates", J Neurosci Methods 420, 110467, doi:10.1016/j.jneumeth.2025.110467.

Dataset: DANDI:002014, version 0.261120.0528 (doi:10.48324/dandi.002014/0.261120.0528), streamed
from the archive. This version was asked for in the prompt of this attempt; it is also the most
recent published version, and the paper cites no DANDI version (its Data availability statement
names the Zenodo record 10.5281/zenodo.14825992). The earlier version, 0.261012.0921, was not read.

Claims: `claims/choi-2025.yaml`, written from the author manuscript in PubMed Central (PMC13250822),
full text. This is a repeat attempt, made blind to the earlier ones.

Outside source: the two trial-id lists of the paper's Zenodo deposit (record 14825992; files
`09272024_nlx_c8c_value object start_big rew and success_tr1-1411.mat`, md5
d3199b1d6d5d7a2e4e2ce2b23d2cb8bb, and `..._small rew and success_tr1-1411.mat`, md5
21b6ec1119ff8924a3dd98a8537bbb68). They are used only to check the trial classes built from the
dandiset's events.

| Claim | Tested here |
|---|---|
| fig8a-single-trial-dopamine | tested |
| fig8b-dopamine-large-vs-small | tested |
| fig9a-spikes-task-aligned | tested |
| fig9bc-unit-waveform-isi | tested |
| fig9d-beta-suppression-large | tested |
| recordings-two-monkeys | blocked, data_missing (monkey J not in the dandiset) |
| scfe-functional-yield | blocked, data_missing |
| scfe-background-yield | blocked, data_missing |
| mcfe-functional-yield | blocked, data_missing |
| dopamine-signal-yield | blocked, data_missing |
| asepsis-cultures | blocked, data_missing |
| fig6-drainage-volume | blocked, data_missing |
| fig7-bone-growth-margins | blocked, data_missing |

In [ ]:
import time, resource, zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import choi2025 as c

T0 = time.time()
assets = c.list_assets()
meta = c.version_metadata()
asset = assets[0]
f = c.open_nwb(asset)
events = c.load_events(f)
trials = c.trial_table(f, events)
fscv = c.load_fscv(f)
units_thr = c.load_units(f, "processing/ecephys/thresholded_units")
units_plx = c.load_units(f, "units")
lfp_t, lfp_x = c.load_lfp(f)
zen = c.zenodo_trial_ids()
print(f"loaded in {time.time() - T0:.0f} s")

## Data check

What was loaded, checked against the descriptions: the archive's metadata, the file's own
descriptions, and the paper. Then whether every stream shares the clock of the trial table.

In [ ]:
print("Dandiset", meta["id"], "|", meta["name"])
print("assets:", [(a["path"], a["size"]) for a in assets])
print("subject:", c.text(f["general/subject/subject_id"][()]), c.text(f["general/subject/sex"][()]),
      c.text(f["general/subject/age"][()]), "| session", c.text(f["general/session_id"][()]),
      "| start", c.text(f["session_start_time"][()]))
el = f["general/extracellular_ephys/electrodes"]
print("electrodes:", [(c.text(n), c.text(l)) for n, l in zip(el["channel_name"][()], el["location"][()])])
print("acquisition:", list(f["acquisition"]), "| processing:", {k: list(f["processing"][k]) for k in f["processing"]})

print("\n-- events --")
print(c.event_descriptions(f).to_string())

print("\n-- trials --")
n_start = len(events["intended trial start"])
print(f"trials table rows: {len(trials)}; trial-start events (code 128): {n_start}; "
      f"first event {events['intended trial start'][0]:.2f} s, first row {trials.start.iloc[0]:.2f} s")
s128 = np.sort(events["intended trial start"])
j = np.clip(np.searchsorted(s128, trials.start.values), 1, len(s128) - 1)
print("largest |midpoint_time - nearest code-128 event|:",
      f"{np.max(np.minimum(np.abs(s128[j] - trials.start.values), np.abs(s128[j - 1] - trials.start.values))):.6f} s")
print(trials.cls.value_counts().to_string())
print("successful trials without V:", int(trials[trials.success].V.isna().sum()),
      "| without RW:", int(trials[trials.success].RW.isna().sum()),
      "| trials with both reward sizes:", int((trials.big & trials.small).sum()))
print("V - trial start (s), successful:", trials[trials.success].eval("V - start").describe()[["min", "50%", "max"]].round(3).to_dict())
print("RW - V (s), successful:", trials[trials.success].eval("RW - V").describe()[["min", "50%", "max"]].round(3).to_dict())
for k in ["large", "small"]:
    ours = set(trials.trial_id[trials.cls == k])
    paper = set(zen[k])
    print(f"{k}: events {len(ours)}, paper's Zenodo list {len(paper)}, in both {len(ours & paper)}, "
          f"events only {sorted(ours - paper)}, list only {sorted(paper - ours)}")
last = trials.iloc[-1]
print(f"last row (id {last.trial_id}): start {last.start:.2f} s, next trial start none, big reward {last.big}, "
      f"error {last.error}, class {last.cls} (open span, left out as in the paper's lists)")

print("\n-- trial-aligned FSCV --")
da = fscv["da"]
print("da shape", da.shape, "| rate", np.unique(fscv["rate"]), "Hz | good flags", int(fscv["good"].sum()),
      "| trial_index == row:", bool(np.all(fscv["trial_index"] == np.arange(len(da)))))
print("window start = midpoint - 30 s:", bool(np.allclose(fscv["start_time"], trials.start - 30)),
      "| stop = midpoint + 30 s:", bool(np.allclose(fscv["stop_time"], trials.start + 30)))
print(f"da range {np.nanmin(da):.2f} to {np.nanmax(da):.2f} (units not given); NaN fraction {np.isnan(da).mean():.3f}; "
      f"rows all NaN {int(np.isnan(da).all(1).sum())}; rows with any NaN {int(np.isnan(da).any(1).sum())}")
print("units attribute of da:", dict(f["processing/fscv/trial_aligned_fscv_table/da"].attrs).get("unit", "none"))

print("\n-- spikes --")
for name, u in [("thresholded", units_thr), ("Plexon", units_plx)]:
    for k, s in u.items():
        print(f"{name} unit {k}: {len(s)} spikes, {s.min():.2f} to {s.max():.2f} s, sorted {bool(np.all(np.diff(s) >= 0))}")
s_thr, s_plx = units_thr["1"], units_plx["ch75#1"]
j = np.clip(np.searchsorted(s_thr, s_plx), 1, len(s_thr) - 1)
d = np.minimum(np.abs(s_thr[j] - s_plx), np.abs(s_thr[j - 1] - s_plx))
print(f"Plexon spikes within 0.1 ms of a thresholded spike: {np.mean(d < 1e-4):.4f}")

print("\n-- LFP --")
dt = np.diff(lfp_t)
print(f"bipolar LFP: {len(lfp_x)} samples, {lfp_t[0]:.2f} to {lfp_t[-1]:.2f} s, median step {np.median(dt)*1e3:.3f} ms, "
      f"steps within 20 us of 1 ms {np.mean(np.abs(dt - 1e-3) < 2e-5):.5f}, NaN {int(np.isnan(lfp_x).sum())}, "
      f"range {np.nanmin(lfp_x)*1e6:.0f} to {np.nanmax(lfp_x)*1e6:.0f} uV")

es = f["acquisition/electrical_series"]
fs_raw = float(es["starting_time"].attrs["rate"])
t_raw0 = float(es["starting_time"][()])
conv = float(es["data"].attrs["conversion"])
print(f"raw ephys: {es['data'].shape}, rate {fs_raw:.4f} Hz, {t_raw0:.2f} to {t_raw0 + es['data'].shape[0]/fs_raw:.2f} s, "
      f"int16 x {conv:.3e} V, so +-{32767*conv*1e3:.2f} mV (headstage input range +-1 mV)")
fts = f["acquisition/fscv_response_series/timestamps"]
scan_t = fts[::214]
v_scan = f["stimulus/presentation/fscv_excitation_series/data"][:214]
i_scan = f["acquisition/fscv_response_series/data"][:214] * f["acquisition/fscv_response_series/data"].attrs["conversion"]
print(f"raw FSCV: {fts.shape[0]} samples = {fts.shape[0] // 214} scans of 214 samples at {1/(fts[1]-fts[0]):.0f} Hz; "
      f"scan interval {np.median(np.diff(scan_t)):.5f} s; scans {scan_t[0]:.2f} to {scan_t[-1]:.2f} s")
print("eye tracking series in the file:", [k for k in f["acquisition"] if "eye" in k.lower()] or "none (the version's release notes mention eye tracking)")
print(f"first scan: voltage {v_scan.min():.2f} to {v_scan.max():.2f} V, current {i_scan.min()*1e9:.0f} to {i_scan.max()*1e9:.0f} nA")

### Clocks

Five checks. (1) The stored bipolar LFP against the raw c7b and c5c traces: the raw difference is
reduced to 1 kHz and both are band-passed; the lag of the best correlation gives the offset
between the raw ephys clock and the trial clock (on which the LFP is stored). The claims file's
opening comment reports an offset of 11812.1 s, found on an earlier version of the dandiset; the
search here covers 5 s either way, so the correlation at that offset is printed too. The spectra
of the stored and raw signals show what the stored LFP keeps in the beta band. (2) The
thresholded spike times against the raw c5d trace: the raw trace, high-passed, averaged at the
spike times. (3) The raw FSCV scan times against the spikes: no spikes fall in about 10 ms of
each FSCV cycle (the file's description), so the phase of that gap within the stored scan times
shows whether the two share a clock; the phase of V within the scan cycle is shown with it.
(4) The trial-aligned oxidation current `iox` (described in the file as the current at about
0.6 V) against the raw scans: a linear kernel from the raw current near 0.6 V to `iox`, fitted
over 36 trials spread over the session, and the lag at which `da` best matches the raw current and
`iox` in those of the 36 trials with no NaN. (5) Event alignment, by contrast: the dopamine difference
between large and small trials around V and around RW, and the spike rate around V; the
random-alignment test of fig9a-spikes-task-aligned is the formal test.


In [ ]:
# (1) stored LFP against raw c7b - c5c
def raw_bipolar(a, b):
    """Raw c7b - c5c from trial-clock time a to b, reduced to 1 kHz (times, volts)."""
    ia, ib = int((a - t_raw0) * fs_raw), int((b - t_raw0) * fs_raw)
    raw = es["data"][ia:ib, :].astype(float) * conv
    bd = signal.decimate(signal.decimate(raw[:, 0] - raw[:, 1], 8), 4)
    return t_raw0 + ia / fs_raw + np.arange(len(bd)) * 32 / fs_raw, bd

rows, spec, far = [], [], []
t_end_raw = t_raw0 + es["data"].shape[0] / fs_raw
for tr in [10, 132, 700, 1300]:
    m = trials.start[tr]
    L = c.lfp_segment(lfp_t, lfp_x, m, 10, 10)
    td, bd = raw_bipolar(m - 15, m + 15)
    R = np.interp(np.arange(m - 15, m + 15, 0.001), td, bd)
    for band in [(1, 8), (13, 30), (60, 200)]:
        sos = signal.butter(4, band, btype="bandpass", fs=1000, output="sos")
        Lf, Rf = signal.sosfiltfilt(sos, L), signal.sosfiltfilt(sos, R)
        lags = np.arange(-5000, 5001)
        Lz = (Lf - Lf.mean()) / Lf.std()
        cc = np.array([np.corrcoef(Lz, Rf[5000 + k:5000 + k + len(Lz)])[0, 1] for k in lags])
        k = int(np.argmax(cc))
        rows.append(dict(trial=tr, band=f"{band[0]}-{band[1]} Hz", best_lag_ms=int(lags[k]),
                         r_best=round(cc[k], 3), r_lag0=round(cc[5000], 3), sd_ratio_stored_raw=round(Lf.std() / Rf.std(), 3)))
    fq, pL = signal.welch(L, fs=1000, nperseg=2000)
    _, pR = signal.welch(R[5000:25000], fs=1000, nperseg=2000)
    spec.append((pL, pR))
    # the claims file's offset of 11812.1 s, in whichever direction the raw recording covers
    sos = signal.butter(4, (1, 8), btype="bandpass", fs=1000, output="sos")
    for sgn in [1, -1]:
        a = m - 10 + sgn * 11812.1
        if a > t_raw0 and a + 20 < t_end_raw:
            tdf, bdf = raw_bipolar(a, a + 20)
            Rx = np.interp(np.arange(a, a + 20, 0.001)[:len(L)], tdf, bdf)
            far.append(dict(trial=tr, shift_s=sgn * 11812.1,
                            r_1_8Hz=round(np.corrcoef(signal.sosfiltfilt(sos, L), signal.sosfiltfilt(sos, Rx))[0, 1], 3)))
xc = pd.DataFrame(rows)
print(xc.to_string(index=False))
low = xc[xc.band == "1-8 Hz"]
offset = float(np.median(low.best_lag_ms)) / 1000  # raw time = trial time + offset
print(f"offset of the raw ephys clock from the trial clock: {offset*1e3:.1f} ms "
      f"(lags {low.best_lag_ms.min()} to {low.best_lag_ms.max()} ms over +-5 s searched, r {low.r_best.min()} to {low.r_best.max()})")
print("correlation (1-8 Hz) with the raw signal moved by the claims file's 11812.1 s:")
print(pd.DataFrame(far).to_string(index=False))

pL = np.mean([s[0] for s in spec], 0); pR = np.mean([s[1] for s in spec], 0)
fsel = [5, 8, 10, 12, 15, 18, 20, 22, 25, 28, 30, 35, 40, 50, 60]
ix = [int(np.argmin(np.abs(fq - x))) for x in fsel]
print("\nWelch spectra over the four trials (V^2/Hz, 0.5 Hz resolution):")
print(pd.DataFrame({"Hz": fsel, "stored": pL[ix], "raw": pR[ix], "stored/raw": pL[ix] / pR[ix]}).to_string(
    index=False, float_format=lambda v: f"{v:.3g}"))
for h in [10, 20, 30, 40]:
    k = int(np.argmin(np.abs(fq - h)))
    nb_ = np.r_[pR[k - 8:k - 2], pR[k + 3:k + 9]].mean()
    nbL = np.r_[pL[k - 8:k - 2], pL[k + 3:k + 9]].mean()
    print(f"{h} Hz: raw {pR[k]/nb_:.2f} x its neighbours (+-1.5 to 4 Hz), stored {pL[k]/nbL:.2f} x")
fig, ax = plt.subplots(figsize=(7, 3))
sel = (fq >= 1) & (fq <= 100)
ax.semilogy(fq[sel], pR[sel], label="raw c7b - c5c"); ax.semilogy(fq[sel], pL[sel], label="stored LFP")
for h in [10, 20, 30]:
    ax.axvline(h, color="k", lw=0.6, ls=":")
ax.set_xlabel("Hz"); ax.set_ylabel("V^2/Hz"); ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# (2) thresholded spike times against the raw c5d trace
t_a = s_thr[5000]
ia = int((t_a - t_raw0) * fs_raw); ib = ia + int(20 * fs_raw)
raw5 = es["data"][ia:ib, 2].astype(float) * conv
hp = signal.sosfiltfilt(signal.butter(4, [300, 6000], btype="bandpass", fs=fs_raw, output="sos"), raw5)
sp = s_thr[(s_thr > t_raw0 + (ia + 100) / fs_raw) & (s_thr < t_raw0 + (ib - 100) / fs_raw)]
idx = np.round((sp - t_raw0) * fs_raw).astype(int) - ia
avg = np.array([hp[i - 40:i + 60] for i in idx]).mean(0)
w_thr = c.thresholded_waveforms(f, 1)
mw = w_thr.mean(0)
print(f"{len(sp)} spikes in 20 s from {t_a:.1f} s: raw c5d trough {np.argmin(avg) - 40} samples after the spike time "
      f"({avg.min()*1e6:.1f} uV); stored waveforms' trough at sample {np.argmin(mw)} of 48 ({mw.min()*1e6:.1f} uV)")

# (3) phase of the spike gap within the stored FSCV scan times
inside = s_thr[(s_thr > scan_t[0]) & (s_thr < scan_t[-1])]
ph = (inside - scan_t[np.searchsorted(scan_t, inside) - 1]) * 1000
gap_rows = []
for a in np.arange(4600, 23000, 2000):
    m = (inside >= a) & (inside < a + 2000)
    if m.sum() < 1000:
        continue
    h = np.histogram(ph[m], bins=np.arange(0, 101, 2))[0]
    gap_rows.append(dict(from_s=a, spikes=int(m.sum()), emptiest_2ms_bin_ms=int(np.argmin(h) * 2),
                         count_there=int(h.min()), median_count=float(np.median(h))))
print(pd.DataFrame(gap_rows).to_string(index=False))
print("phase histogram over the session, 5 ms bins:", np.histogram(ph, bins=np.arange(0, 105, 5))[0].tolist())
succ = trials[trials.success]
v_in = succ.V.values[(succ.V.values > scan_t[0]) & (succ.V.values < scan_t[-1])]
ph_v = (v_in - scan_t[np.searchsorted(scan_t, v_in) - 1]) * 1000
print(f"phase of V within the scan cycle, {len(ph_v)} successful trials, 10 ms bins:",
      np.histogram(ph_v, bins=np.arange(0, 101, 10))[0].tolist())

In [ ]:
# (5) event alignment, by contrast between large and small trials
td_ = c.da_times()
bins = np.arange(-5, 8, 1.0)
rows = []
for i in succ.index:
    v = succ.V[i] - succ.start[i]
    x = da[i] - c.value_at(da[i], td_, v)
    rows.append([c.window_mean(x, td_, v + a, v + a + 1 - 1e-9) for a in bins])
B = pd.DataFrame(rows, index=succ.index, columns=[f"{a:+.0f}..{a+1:+.0f}s" for a in bins])
dd = B[succ.cls == "large"].mean() - B[succ.cls == "small"].mean()
print("dopamine, large minus small, by 1 s window relative to V (referenced at V):")
print(dd.round(2).to_string())
bins_rw = np.arange(-6, 6, 1.0)
rows = []
for i in succ.index:
    rw = succ.RW[i] - succ.start[i]
    x = da[i] - c.value_at(da[i], td_, rw)
    rows.append([c.window_mean(x, td_, rw + a, rw + a + 1 - 1e-9) for a in bins_rw])
B2 = pd.DataFrame(rows, index=succ.index, columns=[f"{a:+.0f}..{a+1:+.0f}s" for a in bins_rw])
print("dopamine by 1 s window relative to RW (referenced at RW): large, small, large minus small")
print(pd.DataFrame({"large": B2[succ.cls == "large"].mean(), "small": B2[succ.cls == "small"].mean(),
                    "large-small": B2[succ.cls == "large"].mean() - B2[succ.cls == "small"].mean()}).round(2).to_string())
edges = np.arange(-1, 3.0001, 0.1)
r = c.psth(s_thr - offset, succ.V.values, edges).mean(0) / 0.1
print("spike rate around V (spikes/s, 100 ms bins from -1 s):", np.round(r, 1).tolist())

In [ ]:
# (4) trial-aligned iox against the raw FSCV current near 0.6 V (anodic sweep, samples 57 to 66)
fsd = f["acquisition/fscv_response_series/data"]
print("excitation at the samples used (V):", np.round(v_scan[57:67], 2).tolist())
K = 40  # lags of -4 to +4 s at 10 Hz
Xs, Ys, best, best_da, best_da_iox = [], [], [], [], []
for i in np.linspace(5, 1405, 36).astype(int):
    m = trials.start[i]
    k0, k1 = np.searchsorted(scan_t, [m - 36, m + 36])
    cur = fsd[k0 * 214:k1 * 214].reshape(-1, 214)[:, 57:67].mean(1)
    st = scan_t[k0:k1]
    y = fscv["iox"][i]; tt_ = td_ + m
    if np.isnan(y).any():
        continue
    cc = [np.corrcoef(signal.detrend(y), signal.detrend(np.interp(tt_ + l / 10, st, cur)))[0, 1] for l in range(-50, 51)]
    best.append((int(np.argmax(cc)) - 50) / 10)
    d = fscv["da"][i]
    if not np.isnan(d).any():
        cd = [np.corrcoef(signal.detrend(d), signal.detrend(np.interp(tt_ + l / 10, st, cur)))[0, 1] for l in range(-50, 51)]
        best_da.append((int(np.argmax(cd)) - 50) / 10)
        ci = [np.corrcoef(signal.detrend(d)[50:-50], signal.detrend(y)[50 + l:len(y) - 50 + l])[0, 1] for l in range(-20, 21)]
        best_da_iox.append((int(np.argmax(ci)) - 20) / 10)
    Xs.append(np.array([np.diff(np.interp(tt_ + l / 10, st, cur)) for l in range(-K, K + 1)]).T)
    Ys.append(np.diff(y))
Xa, Ya = np.vstack(Xs), np.concatenate(Ys)
w = np.linalg.lstsq(Xa / Xa.std(), Ya, rcond=None)[0]
print(f"{len(best)} trials; lag of the best correlation of detrended iox with the raw current at time + lag: "
      f"median {np.median(best):+.1f} s, range {min(best):+.1f} to {max(best):+.1f} s")
print("kernel from the raw current's changes at time + lag to iox's changes (x1e4), lags -4 to +4 s by 0.2 s:")
print({f"{l/10:+.1f}": round(float(v) * 1e4, 1) for l, v in zip(range(-K, K + 1), w) if l % 2 == 0})
print(f"da, {len(best_da)} of these trials without NaN: lag of the best correlation of detrended da with the raw current "
      f"at time + lag: median {np.median(best_da):+.1f} s, range {min(best_da):+.1f} to {max(best_da):+.1f} s; "
      f"with iox at time + lag: median {np.median(best_da_iox):+.1f} s, range {min(best_da_iox):+.1f} to {max(best_da_iox):+.1f} s")
print(f"r of the fitted kernel: {np.corrcoef(Xa / Xa.std() @ w, Ya)[0, 1]:.3f}")


## fig8a-single-trial-dopamine

> In a single trial (trial 133 of the session), dopamine concentration changes at site c8ds
> fluctuate, especially after the value cue display.

Criterion: the `da` trace referenced to its value at V; the SD over 0 to 5 s after V against that
over 5 s to 0 s before V; reproduced when the ratio exceeds 1 under either reading of the trial
(id 133, 1-based, row 132; or row 133). The ratio over all successful trials is a reference.

In [ ]:
rows = []
for i in [132, 133]:
    v = trials.V[i] - trials.start[i]
    x = da[i] - c.value_at(da[i], td_, v)
    after, before = c.window_sd(x, td_, v, v + 5, "right"), c.window_sd(x, td_, v - 5, v, "left")
    rows.append(dict(reading=f"row {i} (trial id {i+1})", cls=trials.cls[i], sd_after=after, sd_before=before,
                     ratio=after / before, nan_after=int(np.isnan(x[(td_ > v) & (td_ <= v + 5)]).sum()),
                     nan_before=int(np.isnan(x[(td_ >= v - 5) & (td_ < v)]).sum())))
fig8a = pd.DataFrame(rows)
print(fig8a.round(3).to_string(index=False))
v = trials.V[132] - trials.start[132]
x = da[132] - c.value_at(da[132], td_, v)
m = (td_ - v >= -5) & (td_ - v <= 5)
print(f"trial id 133 (row 132): da referenced at V, samples from {td_[m][0] - v:+.2f} to {td_[m][-1] - v:+.2f} s after V, 0.1 s apart:")
print(np.round(x[m], 2).tolist())
ratios = []
for i in succ.index:
    v = succ.V[i] - succ.start[i]
    ratios.append(c.window_sd(da[i], td_, v, v + 5, "right") / c.window_sd(da[i], td_, v - 5, v, "left"))
ratios = np.array(ratios)
print(f"reference, all {len(ratios)} successful trials: median ratio {np.nanmedian(ratios):.3f}, "
      f"IQR {np.nanpercentile(ratios, 25):.3f} to {np.nanpercentile(ratios, 75):.3f}, fraction above 1 {np.nanmean(ratios > 1):.3f}, "
      f"NaN {int(np.isnan(ratios).sum())}; trial 133's rank: {np.mean(ratios <= fig8a.ratio[0]):.3f} quantile")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for k, i in enumerate([132, 133]):
    v = trials.V[i] - trials.start[i]
    x = da[i] - c.value_at(da[i], td_, v)
    m = (td_ - v >= -5) & (td_ - v <= 5)
    ax.plot(td_[m] - v, x[m], label=f"row {i} ({trials.cls[i]})")
    print(f"row {i}: plotted range {np.nanmin(x[m]):.2f} to {np.nanmax(x[m]):.2f}")
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("time from V (s)"); ax.set_ylabel("da - da(V) (units not given)")
ax.legend(); plt.tight_layout(); plt.show()

**Result: reproduced.** In trial id 133 (row 132, a large successful trial) the SD of the referenced dopamine trace is 4.414 over 0 to 5 s after V and 2.180 over the 5 s before, a ratio of 2.025; under the other reading (row 133, a small trial) the ratio is 2.199. Both exceed 1. Over the 941 successful trials with a ratio (9 of 950 are NaN) the median ratio is 1.259 and 62.7% are above 1; trial 133 lies at the 0.775 quantile. In trial 133 the referenced trace dips to -1.84 about 1.1 s before V and rises to 13.59 about 1.9 s after it; the paper gives no values to compare with. The trial-aligned FSCV lags the raw scans: `iox` best matches the raw current about 1.5 s earlier, and `da` follows `iox` by a further 0.1 to 0.4 s (data check, 13 trials), so part of the rise after V may come from changes in the raw current just before V.


## fig8b-dopamine-large-vs-small

> Trial-averaged dopamine at site c8ds, aligned to the value cue, increases more for large than
> for small reward trials, especially between the value cue display and reward.

Criterion: per successful trial, `da` referenced at V and averaged from V to RW; large against
small, difference of means above 0 and two-sided Mann-Whitney p < 0.05. Variants: 0 to 2 s,
RW to RW + 3 s, and the two halves of the session.

In [ ]:
rows = {}
for i in succ.index:
    v, rw = succ.V[i] - succ.start[i], succ.RW[i] - succ.start[i]
    x = da[i] - c.value_at(da[i], td_, v)
    rows[i] = dict(V_to_RW=c.window_mean(x, td_, v, rw), V_0_2s=c.window_mean(x, td_, v, v + 2),
                   RW_to_RW3s=c.window_mean(x, td_, rw, rw + 3))
D = pd.DataFrame(rows).T
D["cls"] = succ.cls
half = np.median(trials.start)
D["half"] = np.where(succ.start < half, "first", "second")
out = []
for sel_name, sel in [("all", D.index), ("first half", D.index[D.half == "first"]), ("second half", D.index[D.half == "second"])]:
    for col in ["V_to_RW", "V_0_2s", "RW_to_RW3s"]:
        L = D.loc[sel][D.loc[sel].cls == "large"][col]
        S = D.loc[sel][D.loc[sel].cls == "small"][col]
        U, p = c.mannwhitney(L, S)
        out.append(dict(trials=sel_name, window=col, n_large=int(L.notna().sum()), n_small=int(S.notna().sum()),
                        dropped_nan=int(L.isna().sum() + S.isna().sum()), mean_large=L.mean(), mean_small=S.mean(),
                        diff_means=L.mean() - S.mean(), U=U, p=p))
fig8b = pd.DataFrame(out)
print(fig8b.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
tt = np.arange(-3, 8.01, 0.1)
for k, col in [("large", "C3"), ("small", "C0")]:
    X = []
    for i in succ.index[succ.cls == k]:
        v = succ.V[i] - succ.start[i]
        x = da[i] - c.value_at(da[i], td_, v)
        X.append(np.interp(tt + v, td_, x))
    mu = np.nanmean(X, 0)
    ax.plot(tt, mu, color=col, label=f"{k} (n={len(X)})")
    print(f"{k}: plotted mean range {np.nanmin(mu):.2f} to {np.nanmax(mu):.2f}")
    print(f"  mean at -3 to +8 s from V by 0.5 s:", np.round(mu[::5], 2).tolist())
ax.axvline(0, color="k", lw=0.8); ax.axvline(np.median(succ.RW - succ.V), color="k", ls="--", lw=0.8)
ax.set_xlabel("time from V (s); dashed: median RW"); ax.set_ylabel("da - da(V)"); ax.legend(); plt.tight_layout(); plt.show()

**Result: reproduced.** From V to RW the dopamine trace, referenced at V, averages 1.853 in 448 large and 1.095 in 498 small trials (difference of means 0.758, Mann-Whitney p = 0.0028; 4 trials dropped for NaN). The difference holds from RW to RW + 3 s (1.546, p = 0.00012) but not over 0 to 2 s after V (0.091, p = 0.57); in the data check the large minus small difference is 0.33 or less until 2 s after V, 1.15 to 1.44 from 2 to 4 s, 1.32 from 4 to 5 s and 1.57 to 1.68 from 5 to 8 s (RW is 4.72 to 5.50 s after V). In each half of the session the V to RW difference is positive (0.778, p = 0.051; 0.745, p = 0.018). The criterion tests the V to RW window only. On the statement's "especially between the value cue display and reward": the difference arises between V and RW, rising from 0.33 at 1 to 2 s after V to 1.44 at 3 to 4 s, before RW, and then stays at about 1.6. Referenced at RW, it grows by only 0.19, 0.48 and 0.25 in the 1 s windows over 0 to 3 s after RW. The RW to RW + 3 s window is referenced at V, so it carries the difference built up before RW. The difference thus builds between V and RW and is then held; only its level is as large after RW as before. The trial-aligned FSCV lags the raw scans by about 1.5 s or more (data check), so the null 0 to 2 s variant may in part reflect this delay.


## fig9a-spikes-task-aligned

> Spikes recorded at site c5d on a chamber-implanted CFE can be shown as a raster and a
> peristimulus time histogram aligned to the value cue display, for large and small reward trials.

Criterion: thresholded unit 1 (the unit with nearly all spikes; the Plexon unit is a variant).
(i) At least 90% of successful trials have a spike from 1 s before to 3 s after V; (ii) the
variance across the 100 ms bins from -1 to 3 s of the PSTH exceeds the 95th percentile of the same
statistic over 200 random alignments (V replaced by a uniform time within the same trial's span).
The span is the trial start to the next trial start (the last row, which has no next start, is
not among the successful trials). Spike times are placed on the trial clock with the offset measured above; the
stored times are a variant.

In [ ]:
CLAIM = "fig9a-spikes-task-aligned"
span_end = succ.next_start.values
res = []
null_store = {}
for name, s in [("thresholded, offset applied", s_thr - offset), ("thresholded, stored times", s_thr), ("Plexon ch75#1", s_plx - offset)]:
    cnt = c.psth(s, succ.V.values, np.array([-1.0, 3.0]))[:, 0]
    obs = c.psth_variance(s, succ.V.values, edges)
    for off in [0, 1000, 2000, 3000, 4000]:
        rng = np.random.default_rng(c.seed(CLAIM, 0) + off)
        null = np.array([c.psth_variance(s, rng.uniform(succ.start.values, span_end), edges) for _ in range(200)])
        null_store[(name, off)] = null
        res.append(dict(spikes=name, seed_offset=off, frac_trials_with_spike=np.mean(cnt > 0), var_obs=obs,
                        null_p95=np.percentile(null, 95), null_max=null.max(), frac_null_ge_obs=np.mean(null >= obs)))
fig9a = pd.DataFrame(res)
print(f"{len(succ)} successful trials ({(succ.cls=='large').sum()} large, {(succ.cls=='small').sum()} small)")
print(fig9a.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
R = c.psth(s_thr - offset, succ.V.values, edges) / 0.1
rl, rs = R[succ.cls.values == "large"].mean(0), R[succ.cls.values == "small"].mean(0)
print("PSTH large (spikes/s):", np.round(rl, 1).tolist())
print("PSTH small (spikes/s):", np.round(rs, 1).tolist())
print(f"large minus small: mean over -1..3 s {np.mean(rl - rs):.3f} spikes/s, largest |difference| {np.max(np.abs(rl - rs)):.2f} "
      f"at {edges[np.argmax(np.abs(rl - rs))]:.1f} s; in 0.2..0.6 s: large {rl[12:16].mean():.2f}, small {rs[12:16].mean():.2f}")

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
order = np.r_[np.where(succ.cls.values == "large")[0], np.where(succ.cls.values == "small")[0]]
for row, i in enumerate(order):
    v = succ.V.values[i]
    a, b = np.searchsorted(s_thr - offset, [v - 1, v + 3])
    axs[0].plot((s_thr[a:b] - offset) - v, np.full(b - a, row), "|", ms=1, color="C3" if succ.cls.values[i] == "large" else "C0")
axs[0].set_ylabel("trial (large, then small)")
ctr = edges[:-1] + 0.05
axs[1].plot(ctr, rl, color="C3", label="large"); axs[1].plot(ctr, rs, color="C0", label="small")
axs[1].set_xlabel("time from V (s)"); axs[1].set_ylabel("spikes/s"); axs[1].legend()
print(f"PSTH plotted range {min(rl.min(), rs.min()):.2f} to {max(rl.max(), rs.max()):.2f} spikes/s; raster shows all spikes in -1..3 s")
plt.tight_layout(); plt.show()

**Result: reproduced.** All 950 successful trials have a spike within 1 s before to 3 s after V (fraction 1.0). The variance across bins of the PSTH is 0.4807 (spikes/s)^2, against a 95th percentile of 0.0574 over 200 random alignments under the claim's seed (largest 0.0713), and 0.0574 to 0.0607 under the four other seeds; no random alignment reached the observed value under any seed. The stored spike times (0.4697) and the Plexon unit (0.4829) give the same result. The large and small PSTHs differ by at most 2.03 spikes/s in a bin, with a mean difference of -0.050 spikes/s over -1 to 3 s; the paper states no difference and none is judged. V falls at every phase of the FSCV scan cycle (77 to 108 trials per 10 ms bin), and each 100 ms bin spans one whole 100 ms cycle, so the spike-free gap of each scan cannot make the fall 0.1 to 0.2 s after V or the rise after it.


## fig9bc-unit-waveform-isi

> The spikes at site c5d have a consistent waveform (mean and standard deviation) and an
> interspike interval distribution (1 ms bins) consistent with a single unit.

Criterion: the unit of fig9a; reproduced when fewer than 1% of interspike intervals are shorter
than 1 ms and the largest absolute value of the mean waveform is more than twice the SD across
spikes at that sample; partially when only one holds.

In [ ]:
isi = np.diff(s_thr)
k = int(np.argmax(np.abs(mw)))
sd_k = float(w_thr[:, k].std(ddof=1))
print(f"thresholded unit 1: {len(s_thr)} spikes, {len(isi)} intervals")
print(f"ISI < 1 ms: {np.mean(isi < 0.001):.5f} ({int((isi < 0.001).sum())}); < 2 ms: {np.mean(isi < 0.002):.5f}; "
      f"shortest ISI {isi.min()*1e3:.3f} ms; waveform length {w_thr.shape[1]} samples = {w_thr.shape[1]/32:.2f} ms")
print("ISI histogram, 1 ms bins from 0 to 20 ms:", np.histogram(isi, np.arange(0, 0.0201, 0.001))[0].tolist())
print(f"mean waveform: largest |value| {abs(mw[k])*1e6:.2f} uV at sample {k} (sign {np.sign(mw[k]):+.0f}); "
      f"SD there {sd_k*1e6:.2f} uV; ratio {abs(mw[k])/sd_k:.2f}")
# variant: Plexon unit (no waveforms in the file for it)
isp = np.diff(s_plx)
print(f"variant, Plexon ch75#1: ISI < 1 ms {np.mean(isp < 0.001):.5f} ({int((isp < 0.001).sum())}), shortest {isp.min()*1e3:.3f} ms")
# variant: halves of the session
for nm, m in [("first half", s_thr < half), ("second half", s_thr >= half)]:
    W = w_thr[m]; mm = W.mean(0); kk = int(np.argmax(np.abs(mm)))
    print(f"{nm}: {m.sum()} spikes, peak {mm[kk]*1e6:.2f} uV, SD {W[:, kk].std(ddof=1)*1e6:.2f} uV, ratio {abs(mm[kk])/W[:, kk].std(ddof=1):.2f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
tw = np.arange(w_thr.shape[1]) / 32
sdw = w_thr.std(0, ddof=1)
axs[0].plot(tw, mw * 1e6, "k"); axs[0].fill_between(tw, (mw - sdw) * 1e6, (mw + sdw) * 1e6, alpha=0.3)
axs[0].set_xlabel("ms"); axs[0].set_ylabel("uV")
axs[1].hist(isi * 1e3, bins=np.arange(0, 101, 1)); axs[1].set_xlabel("ISI (ms)")
print(f"waveform plotted range {(mw - sdw).min()*1e6:.1f} to {(mw + sdw).max()*1e6:.1f} uV; "
      f"ISI histogram shows 0-100 ms, {int((isi > 0.1).sum())} intervals ({np.mean(isi > 0.1):.3f}) beyond")
plt.tight_layout(); plt.show()

**Result: reproduced.** None of the 101732 interspike intervals is shorter than 1 ms (the shortest is 1.562 ms), and the mean waveform's largest absolute value, 56.36 uV at sample 10, is 4.79 times the SD across spikes at that sample (11.76 uV); the ratio is 4.78 and 5.05 in the two halves of the session. The refractory test cannot fail on these spikes: the shortest interval is about the waveform length (48 samples, 1.50 ms), so the detector may not report a second spike within one waveform. The Plexon unit has 2 intervals shorter than 1 ms (fraction 0.00002). Both tests of the criterion hold; they do not show that the spikes come from a single neuron, which the paper itself does not state (a question to a reader is recorded).


## fig9d-beta-suppression-large

> Trial-averaged beta-band LFP power at c7b referenced to c5c is suppressed more in the brief
> window after the value cue for large than for small reward cues.

Criterion: the stored bipolar LFP band-passed to 13 to 30 Hz with a 4th order zero-phase
Butterworth filter over V - 3 s to V + 3 s; power is the squared Hilbert envelope; per successful
trial log10 of mean power over 0 to 0.5 s after V over mean power over 0.5 s before V. Reproduced
when large is lower than small (difference of medians below 0) with a two-sided Mann-Whitney
p < 0.05; partially when both medians are below 0 and large is lower but p >= 0.05. Variants:
windows of 0 to 0.3 s and 0 to 1 s, a 1 s baseline, bands of 15 to 30 and 12 to 25 Hz. The
filter is designed with N = 4 (scipy's convention, a standard choice); N = 2 is shown too.

In [ ]:
segs = np.array([c.lfp_segment(lfp_t, lfp_x, v, 3, 3) for v in succ.V.values])
print(f"segments: {segs.shape}, with NaN: {int(np.isnan(segs).any(1).sum())}")
def beta_table(order):
    out = []
    for name, band, win, base in [("main", (13, 30), 0.5, 0.5), ("window 0-0.3 s", (13, 30), 0.3, 0.5),
                                  ("window 0-1 s", (13, 30), 1.0, 0.5), ("baseline 1 s", (13, 30), 0.5, 1.0),
                                  ("band 15-30 Hz", (15, 30), 0.5, 0.5), ("band 12-25 Hz", (12, 25), 0.5, 0.5)]:
        m = np.array([np.nan if np.isnan(sg).any() else
                      np.log10(c.band_power(sg, band, order=order)[3000:3000 + int(win * 1000)].mean() /
                               c.band_power(sg, band, order=order)[3000 - int(base * 1000):3000].mean()) for sg in segs])
        L, S = m[succ.cls.values == "large"], m[succ.cls.values == "small"]
        U, p = c.mannwhitney(L, S)
        out.append(dict(filter_N=order, variant=name, n_large=int(np.isfinite(L).sum()), n_small=int(np.isfinite(S).sum()),
                        median_large=np.nanmedian(L), median_small=np.nanmedian(S),
                        diff_medians=np.nanmedian(L) - np.nanmedian(S), U=U, p=p))
    return pd.DataFrame(out)
fig9d = pd.concat([beta_table(4), beta_table(2)], ignore_index=True)
print(fig9d.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
# halves of the session, main measure
m_main = np.array([c.beta_suppression(sg, 3, (13, 30), 0.5, 0.5) for sg in segs])
for nm, sel in [("first half", succ.start.values < half), ("second half", succ.start.values >= half)]:
    L, S = m_main[sel & (succ.cls.values == "large")], m_main[sel & (succ.cls.values == "small")]
    print(f"{nm}: median large {np.median(L):.4f}, small {np.median(S):.4f}, p {c.mannwhitney(L, S)[1]:.3g}")

In [ ]:
P = np.array([c.band_power(sg, (13, 30)) for sg in segs])
base = P[:, 2500:3000].mean(1, keepdims=True)
tt = np.arange(-3, 3, 0.001)
fig, ax = plt.subplots(figsize=(7, 3))
for k, col in [("large", "C3"), ("small", "C0")]:
    mu = (P[succ.cls.values == k] / base[succ.cls.values == k]).mean(0)
    sel = (tt >= -1) & (tt <= 2)
    ax.plot(tt[sel], mu[sel], color=col, label=k)
    print(f"{k}: plotted relative power range {mu[sel].min():.3f} to {mu[sel].max():.3f}; "
          f"mean over 0 to 0.5 s after V {mu[3000:3500].mean():.3f}")
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("time from V (s)"); ax.set_ylabel("13-30 Hz power / baseline"); ax.legend()
plt.tight_layout(); plt.show()

**Result: partially reproduced.** Beta power falls after V in both conditions: the median log10 ratio is -0.156 in 451 large and -0.125 in 499 small trials. Large is lower (difference of medians -0.031), but the two-sided Mann-Whitney test gives p = 0.112, above 0.05. No variant with the N = 4 filter reaches p < 0.05 (p 0.066 to 0.83); with N = 2 the 15 to 30 Hz band gives p = 0.052 and the other variants 0.11 to 0.81. Over 0 to 0.3 s after V the medians do not differ (difference -0.0001, p = 0.59). The trial-averaged power ratio plotted above dips lower for large trials (to 0.532, against 0.695; mean over 0 to 0.5 s after V 0.841, against 0.885), which resembles the paper's figure; the plot averages power ratios over trials, and the criterion compares the medians of the per-trial log ratios.


## Blocked claims

The dandiset holds one session of monkey T (the session of Figures 8 and 9). The electrode yields
(scfe-functional-yield, scfe-background-yield, mcfe-functional-yield, dopamine-signal-yield), the
bacterial cultures (asepsis-cultures), the drainage volumes (fig6-drainage-volume), the bone
growth and margin photographs (fig7-bone-growth-margins), and all recordings of monkey J
(recordings-two-monkeys) are not in it, nor in the paper's Zenodo record, whose six files are
listed below; no other deposit was found on DANDI or Zenodo.

In [ ]:
print(c.zenodo_files().to_string(index=False))
print("subjects in the dandiset:", sorted({a["path"].split("/")[0] for a in assets}))
print(f"total runtime {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| fig8a-single-trial-dopamine | reproduced |
| fig8b-dopamine-large-vs-small | reproduced |
| fig9a-spikes-task-aligned | reproduced |
| fig9bc-unit-waveform-isi | reproduced (the refractory test cannot fail on these spikes) |
| fig9d-beta-suppression-large | partially reproduced (both suppressed, large lower, p = 0.112) |
| recordings-two-monkeys | blocked, data_missing |
| scfe-functional-yield, scfe-background-yield, mcfe-functional-yield, dopamine-signal-yield | blocked, data_missing |
| asepsis-cultures, fig6-drainage-volume, fig7-bone-growth-margins | blocked, data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper describes none of the analyses of Figures 8 and 9 beyond the legends, and reports no statistical test. The windows, references, tests, the beta band and the filter come from the claims file's criteria. Trials are classed from the dandiset's events, leaving out the last row (id 1411), which has no next trial start; this gives the paper's Zenodo lists of large (451) and small (499) successful trials exactly. The dopamine values are the stored PCA estimates (units not given), referenced at V by subtraction in place of the paper's background subtraction of the voltammograms at V. Spikes are the stored threshold crossings, moved by the 0.5 ms clock offset measured in the data check.

What was found about the dataset. The dandiset holds one session of monkey T, the session of Figures 8 and 9; the surgical, culture, drainage and yield results and monkey J are not in it. All streams share one clock: the stored LFP matches the raw c7b minus c5c at a lag of 0 to 1 ms (and not at the 11812.1 s the claims file reports for an earlier version, r -0.12 to -0.02), the thresholded spikes fall on the raw c5d troughs, and the spike-free gap sits 46 to 50 ms into every stored FSCV scan interval over the whole session. The trial-aligned oxidation current follows the raw current near 0.6 V as a smoothing over about the preceding 2.8 s: the fitted kernel is positive from about -2.8 to +0.2 s, largest at -0.2 to 0 s, and negative from +0.4 s. Its peak is at lag 0, but the detrended oxidation current matches the raw current best about 1.5 s earlier (0.8 to 2.5 s), so responses in it appear late; the PCA dopamine estimate follows the oxidation current by a further 0.1 to 0.4 s. This smoothing is not documented. The dopamine difference between large and small trials is 0.33 or less until 2 s after V, grows before RW and is then held, and the spike rate is modulated at V. The PCA components have no units. The raw c7b minus c5c carries lines at the FSCV scan rate, 10 Hz, and its harmonics (440 to 2200 times their neighbours at 10 to 40 Hz); the stored LFP has these lines removed and otherwise equals the raw signal (spectral ratio 1.03 to 1.05 at 5 to 35 Hz between the lines), a preprocessing the file says is undocumented. The thresholded unit and the Plexon unit hold the same spikes (counted once here). The version's release notes mention eye tracking, which the file does not hold.
